## Generation of tables and figures for report and appendix 

## Imports and PATH setup

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests


import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path("..")
ROOT / "data"
RESULTS_DIR = ROOT / "results"
OUTPUT_DIR = RESULTS_DIR / "report_tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_NAME = "local_1260_260722"

### Table 1: Dataset structural and morphological stratification

In [ ]:
manifest = pd.read_csv(
    RESULTS_DIR / "phase5_local_1260_260722_resolved_manifest.csv"
)

size_order = ["normal", "small", "tiny", "very_tiny"]

size_labels = {
    "normal": r"Normal ($\geq 5\%$)",
    "small": r"Small ($2\%-<5\%$)",
    "tiny": r"Tiny ($1\%-<2\%$)",
    "very_tiny": r"Very Tiny ($<1\%$)",
}

table_1 = (
    manifest["mask_size_class"]
    .value_counts()
    .reindex(size_order)
    .reset_index()
)

table_1.columns = ["size_class", "count"]

table_1["dataset_percentage"] = (
    table_1["count"] / table_1["count"].sum() * 100
).round(1)

table_1["scale_tier"] = table_1["size_class"].map(size_labels)

table_1 = table_1[
    ["scale_tier", "count", "dataset_percentage"]
]

table_1.to_csv(
    OUTPUT_DIR / "table_1_dataset_size_stratification.csv",
    index=False,
)

display(table_1)

### Table II: Baseline classifier validation results

In [ ]:

model_metrics = pd.read_csv(
    RESULTS_DIR / "phase2_resnet50_validation_metrics.csv"
)

row = model_metrics.iloc[0]

table_2 = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1",
        "AUC-ROC",
    ],
    "Value": [
        row["accuracy"],
        row["macro_precision"],
        row["macro_recall"],
        row["macro_f1"],
        row["auc_roc"],
    ],
})

table_2["Value"] = table_2["Value"].round(3)

display(table_2)

### Table III: Lesion-level mask overlap summary

In [ ]:
lesion_summary = pd.read_csv(
    RESULTS_DIR / "phase5_local_1260_260722_mask_summary_by_method.csv"
)

method_names = {
    "gradcam": "Grad-CAM",
    "lime": "LIME",
    "shap": "SHAP",
}

table_3 = lesion_summary[
    [
        "xai_method",
        "mean_iou",
        "mean_dice",
        "mean_sir",
        "mean_tixai",
    ]
].copy()

table_3["xai_method"] = table_3["xai_method"].map(method_names)

table_3.columns = [
    "Method",
    "Mean IoU",
    "Mean Dice",
    "Mean SIR",
    "Mean TIxAI",
]

table_3 = table_3.round(3)

display(table_3)

### Table IV: Mean Pseudo-Concept Area as a Percentage of Lesion Area

In [ ]:
concept_metrics = pd.read_csv(
    RESULTS_DIR / "phase5_local_1260_260722_xai_concept_metrics.csv"
)

manifest_labels = pd.read_csv(
    RESULTS_DIR / "phase5_local_1260_260722_resolved_manifest.csv"
)[["dataset", "stem", "label_name"]]

selected_concepts = {
    "asymmetry": "Asymmetry",
    "border_w16_dil6_sigma10_dist": "Border Irregularity",
    "colour_heterogeneity": "Colour Heterogeneity",
}

concept_areas = (
    concept_metrics[
        concept_metrics["concept"].isin(selected_concepts)
    ][
        ["dataset", "stem", "concept", "concept_area_ratio"]
    ]
    .drop_duplicates(["dataset", "stem", "concept"])
    .merge(
        manifest_labels,
        on=["dataset", "stem"],
        how="left",
    )
)

rows = []

for concept, display_name in selected_concepts.items():
    concept_data = concept_areas[
        concept_areas["concept"] == concept
    ]

    melanoma_mean = concept_data[
        (concept_data["dataset"] == "ham10000")
        & (concept_data["label_name"] == "melanoma")
    ]["concept_area_ratio"].mean() * 100

    non_melanoma_mean = concept_data[
        (concept_data["dataset"] == "ham10000")
        & (concept_data["label_name"] == "non-melanoma")
    ]["concept_area_ratio"].mean() * 100

    overall_mean = (
        concept_data["concept_area_ratio"].mean() * 100
    )

    rows.append({
        "Concept": display_name,
        "Melanoma": melanoma_mean,
        "Non-Melanoma": non_melanoma_mean,
        "Overall": overall_mean,
    })

table_4 = pd.DataFrame(rows).round(2)

display(table_4)

### Table V: Best mean Dice scores by evaluation target

In [ ]:

lesion_summary = pd.read_csv(
    RESULTS_DIR / "phase5_local_1260_260722_mask_summary_by_method.csv"
)

concept_summary = pd.read_csv(
    RESULTS_DIR / "phase5_local_1260_260722_summary_by_method_concept.csv"
)

selected_concepts = {
    "asymmetry": "Asymmetry",
    "border_w16_dil6_sigma10_dist": "Border Irregularity",
    "colour_heterogeneity": "Colour Heterogeneity",
}

method_names = {
    "gradcam": "Grad-CAM",
    "lime": "LIME",
    "shap": "SHAP",
}

rows = []

# Best lesion-level Dice
best_lesion = lesion_summary.loc[
    lesion_summary["mean_dice"].idxmax()
]

rows.append({
    "Evaluation Target": "Lesion Mask",
    "Best Method": method_names[best_lesion["xai_method"]],
    "Mean Dice": best_lesion["mean_dice"],
})

# Best Dice for each selected pseudo-concept
for concept, concept_name in selected_concepts.items():

    concept_data = concept_summary[
        concept_summary["concept"] == concept
    ]

    best_row = concept_data.loc[
        concept_data["mean_dice"].idxmax()
    ]

    rows.append({
        "Evaluation Target": concept_name,
        "Best Method": method_names[best_row["xai_method"]],
        "Mean Dice": best_row["mean_dice"],
    })

table_5 = pd.DataFrame(rows)

table_5["Mean Dice"] = table_5["Mean Dice"].round(3)

display(table_5)

### Table VI: Representative Differences in Pseudo-Concept Alignment Between Melanoma and Non-Melanoma Lesions

In [ ]:
concept_metrics = pd.read_csv(
    RESULTS_DIR / "phase5_local_1260_260722_xai_concept_metrics.csv"
)

labels = pd.read_csv(
    RESULTS_DIR / "phase5_local_1260_260722_resolved_manifest.csv"
)[["dataset", "stem", "label_name"]]

metrics_with_labels = concept_metrics.merge(
    labels,
    on=["dataset", "stem"],
    how="left",
)

metrics_with_labels = metrics_with_labels[
    metrics_with_labels["dataset"] == "ham10000"
].copy()

selected_concepts = {
    "asymmetry": "Asymmetry",
    "border_w16_dil6_sigma10_dist": "Border Irregularity",
    "colour_heterogeneity": "Colour Heterogeneity",
}

selected_metrics = ["dice", "sir", "tixai"]

rows = []

for method in ["gradcam", "lime", "shap"]:
    for concept, concept_name in selected_concepts.items():
        for metric in selected_metrics:

            subset = metrics_with_labels[
                (metrics_with_labels["xai_method"] == method)
                & (metrics_with_labels["concept"] == concept)
            ]

            melanoma = subset[
                subset["label_name"] == "melanoma"
            ][metric].dropna()

            non_melanoma = subset[
                subset["label_name"] == "non-melanoma"
            ][metric].dropna()

            u_stat, p_value = mannwhitneyu(
                melanoma,
                non_melanoma,
                alternative="two-sided",
            )

            effect = (
                2 * u_stat / (len(melanoma) * len(non_melanoma))
                - 1
            )

            rows.append({
                "Method": method,
                "Concept": concept_name,
                "Metric": metric,
                "Melanoma Mean": melanoma.mean(),
                "Non-Melanoma Mean": non_melanoma.mean(),
                "Effect": effect,
                "p_value": p_value,
            })

all_tests = pd.DataFrame(rows)

all_tests["FDR p"] = multipletests(
    all_tests["p_value"],
    method="fdr_bh",
)[1]

method_names = {
    "gradcam": "Grad-CAM",
    "lime": "LIME",
    "shap": "SHAP",
}

metric_names = {
    "dice": "Dice",
    "sir": "SIR",
    "tixai": "TIxAI",
}

all_tests["Method"] = all_tests["Method"].map(method_names)
all_tests["Metric"] = all_tests["Metric"].map(metric_names)

# Representative rows used in Table VI
representative_rows = [
    ("Grad-CAM", "Colour Heterogeneity", "SIR"),
    ("Grad-CAM", "Border Irregularity", "TIxAI"),
    ("Grad-CAM", "Asymmetry", "TIxAI"),
    ("LIME", "Asymmetry", "Dice"),
    ("SHAP", "Colour Heterogeneity", "SIR"),
]

table_6 = pd.concat([
    all_tests[
        (all_tests["Method"] == method)
        & (all_tests["Concept"] == concept)
        & (all_tests["Metric"] == metric)
    ]
    for method, concept, metric in representative_rows
])

table_6 = table_6[
    [
        "Method",
        "Concept",
        "Metric",
        "Melanoma Mean",
        "Non-Melanoma Mean",
        "Effect",
        "FDR p",
    ]
].reset_index(drop=True)

table_6[
    [
        "Melanoma Mean",
        "Non-Melanoma Mean",
        "Effect",
        "FDR p",
    ]
] = table_6[
    [
        "Melanoma Mean",
        "Non-Melanoma Mean",
        "Effect",
        "FDR p",
    ]
].round(3)

display(table_6)

### Table VII: Mean Dice Alignment Across Lesion-Size Tiers for Asymmetry and Distance-Weighted Border Irregularity

In [ ]:
size_summary = pd.read_csv(
    RESULTS_DIR / "phase5_local_1260_260722_summary_by_size_class.csv"
)

selected_concepts = [
    "asymmetry",
    "border_w16_dil6_sigma10_dist",
]

size_order = [
    "normal",
    "small",
    "tiny",
    "very_tiny",
]

size_names = {
    "normal": "Normal",
    "small": "Small",
    "tiny": "Tiny",
    "very_tiny": "Very Tiny",
}

method_names = {
    "gradcam": "Grad-CAM",
    "lime": "LIME",
    "shap": "SHAP",
}

concept_names = {
    "asymmetry": "Asymmetry",
    "border_w16_dil6_sigma10_dist": "Border",
}

table_7 = (
    size_summary[
        size_summary["concept"].isin(selected_concepts)
    ]
    .pivot(
        index="mask_size_class",
        columns=["concept", "xai_method"],
        values="mean_dice",
    )
    .reindex(size_order)
)

table_7.columns = [
    f"{concept_names[concept]} {method_names[method]}"
    for concept, method in table_7.columns
]

column_order = [
    "Asymmetry Grad-CAM",
    "Asymmetry LIME",
    "Asymmetry SHAP",
    "Border Grad-CAM",
    "Border LIME",
    "Border SHAP",
]

table_7 = table_7[column_order].reset_index()

table_7["mask_size_class"] = (
    table_7["mask_size_class"].map(size_names)
)

table_7 = table_7.rename(
    columns={"mask_size_class": "Size Tier"}
)

table_7.iloc[:, 1:] = table_7.iloc[:, 1:].round(4)

display(table_7)

### Table VIII: Raw Dice and chance-adjusted Dice enrichment

In [ ]:
chance_summary = pd.read_csv(
    RESULTS_DIR / "phase5_local_1260_260722_chance_summary_by_method_concept.csv"
)

selected_concepts = {
    "asymmetry": "Asymmetry",
    "border_w16_dil6_sigma10_dist": "Border Irregularity",
    "colour_heterogeneity": "Colour Heterogeneity",
}

method_names = {
    "gradcam": "Grad-CAM",
    "lime": "LIME",
    "shap": "SHAP",
}

table_8 = chance_summary[
    chance_summary["concept"].isin(selected_concepts)
][
    [
        "xai_method",
        "concept",
        "mean_observed_dice",
        "mean_dice_enrichment",
    ]
].copy()

table_8["xai_method"] = table_8["xai_method"].map(method_names)
table_8["concept"] = table_8["concept"].map(selected_concepts)

table_8.columns = [
    "Method",
    "Concept",
    "Raw Dice",
    "Dice Enrichment",
]

method_order = ["Grad-CAM", "LIME", "SHAP"]
concept_order = [
    "Asymmetry",
    "Border Irregularity",
    "Colour Heterogeneity",
]

table_8["Method"] = pd.Categorical(
    table_8["Method"],
    categories=method_order,
    ordered=True,
)

table_8["Concept"] = pd.Categorical(
    table_8["Concept"],
    categories=concept_order,
    ordered=True,
)

table_8 = (
    table_8
    .sort_values(["Method", "Concept"])
    .reset_index(drop=True)
)

table_8["Raw Dice"] = table_8["Raw Dice"].round(3)
table_8["Dice Enrichment"] = table_8["Dice Enrichment"].round(3)

display(table_8)


### Graphical representation of lesion-size sensitivity for selected concepts and metrics
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# Figure: chance-adjusted Dice enrichment by XAI method
# ============================================================

plot_df = table_8.copy()

# Convert categorical columns back to strings for plotting
plot_df["Method"] = plot_df["Method"].astype(str)
plot_df["Concept"] = plot_df["Concept"].astype(str)

# Pivot so:
# rows    = concepts
# columns = XAI methods
# values  = Dice enrichment
pivot = (
    plot_df
    .pivot(
        index="Concept",
        columns="Method",
        values="Dice Enrichment",
    )
    .reindex(concept_order)
    .reindex(columns=method_order)
)

fig, ax = plt.subplots(figsize=(9, 5))

x = np.arange(len(pivot.index))
width = 0.24

for method_index, method in enumerate(method_order):
    offset = (method_index - 1) * width

    bars = ax.bar(
        x + offset,
        pivot[method].values,
        width=width,
        label=method,
    )

    ax.bar_label(
        bars,
        fmt="%.3f",
        padding=3,
        fontsize=9,
    )

# Random-overlap reference line
ax.axhline(
    y=1.0,
    linestyle="--",
    linewidth=1.5,
    label="Random baseline",
)

ax.set_xticks(x)
ax.set_xticklabels(pivot.index)

ax.set_title(
    "Chance-Adjusted Dice Enrichment by XAI Method",
    fontsize=14,
)

ax.set_xlabel("Pseudo-concept")
ax.set_ylabel("Dice enrichment")

ax.set_ylim(
    0,
    max(1.30, pivot.max().max() + 0.12),
)

ax.grid(axis="y", alpha=0.3)
ax.legend()

fig.tight_layout()


### Fig. 3: Mean pseudo-concept alignment across XAI methods - Dice and TIxAI 

In [ ]:
SUMMARY_CSV = (RESULTS_DIR / "phase5_local_1260_260722_summary_by_method_concept.csv")

OUTPUT_PATH = ( RESULTS_DIR / "fig_3_pseudo_concept_dice_tixai_heatmaps.png"
)

METHOD_ORDER = ["gradcam", "shap", "lime"]

METHOD_LABELS = {"gradcam": "Grad-CAM", "shap": "SHAP", "lime": "LIME"}

CONCEPT_ORDER = ["asymmetry", "border_w16_dil6_sigma10_dist","colour_heterogeneity"]

CONCEPT_LABELS = {"asymmetry": "Asymmetry", "border_w16_dil6_sigma10_dist": "Border irregularity",
                 "colour_heterogeneity": "Colour heterogeneity"}

summary = pd.read_csv(SUMMARY_CSV)
plot_data = summary[
    summary["xai_method"].isin(METHOD_ORDER)
    & summary["concept"].isin(CONCEPT_ORDER)
].copy()

plot_data["xai_method"] = pd.Categorical(
    plot_data["xai_method"],
    categories=METHOD_ORDER,
    ordered=True,
)

plot_data["concept"] = pd.Categorical(
    plot_data["concept"],
    categories=CONCEPT_ORDER,
    ordered=True,
)

plot_data = plot_data.sort_values(
    ["concept", "xai_method"]
)

# ------------------------------------------------------------
# Construct matrices


dice_matrix = (
    plot_data
    .pivot(
        index="concept",
        columns="xai_method",
        values="mean_dice",
    )
    .reindex(
        index=CONCEPT_ORDER,
        columns=METHOD_ORDER,
    )
)

tixai_matrix = (
    plot_data
    .pivot(
        index="concept",
        columns="xai_method",
        values="mean_tixai",
    )
    .reindex(
        index=CONCEPT_ORDER,
        columns=METHOD_ORDER,
    )
)

if dice_matrix.isna().any().any():
    raise ValueError(
        "Missing Dice values after filtering. "
        "Check the concept and method names."
    )

if tixai_matrix.isna().any().any():
    raise ValueError(
        "Missing TIxAI values after filtering. "
        "Check the concept and method names."
    )


# ------------------------------------------------------------
# Heatmap helper

def draw_heatmap(
    ax,
    matrix,
    title,
    colourbar_label,
    decimals=3,
):
    values = matrix.to_numpy(dtype=float)

    image = ax.imshow(
        values,
        aspect="auto",
        interpolation="nearest",
    )

    ax.set_title(
        title,
        fontsize=10,
        pad=8,
    )

    ax.set_xticks(
        np.arange(len(METHOD_ORDER)),
        labels=[
            METHOD_LABELS[method]
            for method in METHOD_ORDER
        ],
    )

    ax.set_yticks(
        np.arange(len(CONCEPT_ORDER)),
        labels=[
            CONCEPT_LABELS[concept]
            for concept in CONCEPT_ORDER
        ],
    )

    ax.tick_params(
        axis="x",
        labelsize=8,
    )

    ax.tick_params(
        axis="y",
        labelsize=8,
    )

    threshold = (
        values.min()
        + values.max()
    ) / 2

    for row_index in range(values.shape[0]):
        for column_index in range(values.shape[1]):
            value = values[row_index, column_index]

            text_colour = (
                "white"
                if value < threshold
                else "black"
            )

            ax.text(
                column_index,
                row_index,
                f"{value:.{decimals}f}",
                ha="center",
                va="center",
                fontsize=8,
                color=text_colour,
            )

    colourbar = ax.figure.colorbar(
        image,
        ax=ax,
        fraction=0.046,
        pad=0.04,
    )

    colourbar.set_label(
        colourbar_label,
        fontsize=8,
    )

    colourbar.ax.tick_params(
        labelsize=7,
    )

# ------------------------------------------------------------
# Save Dice heatmap

DICE_OUTPUT_PATH = (
    RESULTS_DIR
    / "fig_pseudo_concept_dice_heatmap.png"
)

fig, ax = plt.subplots(figsize=(5.2, 3.8))

draw_heatmap(
    ax,
    dice_matrix,
    title="Pseudo-concept alignment by method: Dice",
    colourbar_label="Mean Dice",
)

fig.tight_layout()

fig.savefig(
    DICE_OUTPUT_PATH,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print("Saved Dice heatmap:", DICE_OUTPUT_PATH)


# ============================================================
# Save TIxAI heatmap


TIXAI_OUTPUT_PATH = (
    RESULTS_DIR
    / "fig_pseudo_concept_tixai_heatmap.png"
)

fig, ax = plt.subplots(figsize=(5.2, 3.8))

draw_heatmap(
    ax,
    tixai_matrix,
    title="Pseudo-concept alignment by method: TIxAI",
    colourbar_label="Mean TIxAI",
)

fig.tight_layout()

fig.savefig(
    TIXAI_OUTPUT_PATH,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print("Saved TIxAI heatmap:", TIXAI_OUTPUT_PATH)

## Appendix Tables 

### 3.1: Complete size-stratified metric results

In [ ]:
size_summary = pd.read_csv(
    RESULTS_DIR / "phase5_local_1260_260722_summary_by_size_class.csv"
)

selected_concepts = {
    "asymmetry": "Asymmetry",
    "border_w16_dil6_sigma10_dist": "Border Irregularity",
}

method_names = {
    "gradcam": "Grad-CAM",
    "lime": "LIME",
    "shap": "SHAP",
}

size_names = {
    "normal": "Normal",
    "small": "Small",
    "tiny": "Tiny",
    "very_tiny": "Very Tiny",
}

size_order = ["normal", "small", "tiny", "very_tiny"]

appendix_size_all_metrics = size_summary[
    size_summary["concept"].isin(selected_concepts)
][
    [
        "mask_size_class",
        "xai_method",
        "concept",
        "mean_iou",
        "mean_dice",
        "mean_sir",
        "mean_tixai",
    ]
].copy()

appendix_size_all_metrics["Size Tier"] = (
    appendix_size_all_metrics["mask_size_class"].map(size_names)
)

appendix_size_all_metrics["Method"] = (
    appendix_size_all_metrics["xai_method"].map(method_names)
)

appendix_size_all_metrics["Concept"] = (
    appendix_size_all_metrics["concept"].map(selected_concepts)
)

appendix_size_all_metrics["size_order"] = (
    appendix_size_all_metrics["mask_size_class"]
    .map({name: index for index, name in enumerate(size_order)})
)

appendix_size_all_metrics = (
    appendix_size_all_metrics
    .sort_values(["Concept", "size_order", "Method"])
    [
        [
            "Concept",
            "Size Tier",
            "Method",
            "mean_iou",
            "mean_dice",
            "mean_sir",
            "mean_tixai",
        ]
    ]
    .rename(
        columns={
            "mean_iou": "Mean IoU",
            "mean_dice": "Mean Dice",
            "mean_sir": "Mean SIR",
            "mean_tixai": "Mean TIxAI",
        }
    )
)

appendix_size_all_metrics.iloc[:, 3:] = (
    appendix_size_all_metrics.iloc[:, 3:].round(4)
)

display(appendix_size_all_metrics)

### 3.2: Colour heterogeneity by lesion-size tier

In [ ]:
colour_size = size_summary[
    size_summary["concept"] == "colour_heterogeneity"
][
    [
        "mask_size_class",
        "xai_method",
        "mean_iou",
        "mean_dice",
        "mean_sir",
        "mean_tixai",
    ]
].copy()

colour_size["Size Tier"] = (
    colour_size["mask_size_class"].map(size_names)
)

colour_size["Method"] = (
    colour_size["xai_method"].map(method_names)
)

colour_size["size_order"] = (
    colour_size["mask_size_class"]
    .map({name: index for index, name in enumerate(size_order)})
)

appendix_colour_by_size = (
    colour_size
    .sort_values(["size_order", "Method"])
    [
        [
            "Size Tier",
            "Method",
            "mean_iou",
            "mean_dice",
            "mean_sir",
            "mean_tixai",
        ]
    ]
    .rename(
        columns={
            "mean_iou": "Mean IoU",
            "mean_dice": "Mean Dice",
            "mean_sir": "Mean SIR",
            "mean_tixai": "Mean TIxAI",
        }
    )
)

appendix_colour_by_size.iloc[:, 2:] = (
    appendix_colour_by_size.iloc[:, 2:].round(4)
)

display(appendix_colour_by_size)

### 3.3: Chance-adjusted Dice enrichment by size tier

In [ ]:
chance_size = pd.read_csv(
    RESULTS_DIR / "phase5_local_1260_260722_chance_summary_by_size_class.csv"
)

chance_concepts = {
    "asymmetry": "Asymmetry",
    "border_w16_dil6_sigma10_dist": "Border Irregularity",
    "colour_heterogeneity": "Colour Heterogeneity",
}

appendix_chance_by_size = chance_size[
    chance_size["concept"].isin(chance_concepts)
][
    [
        "mask_size_class",
        "xai_method",
        "concept",
        "mean_observed_dice",
        "mean_dice_enrichment",
    ]
].copy()

appendix_chance_by_size["Size Tier"] = (
    appendix_chance_by_size["mask_size_class"].map(size_names)
)

appendix_chance_by_size["Method"] = (
    appendix_chance_by_size["xai_method"].map(method_names)
)

appendix_chance_by_size["Concept"] = (
    appendix_chance_by_size["concept"].map(chance_concepts)
)

appendix_chance_by_size["size_order"] = (
    appendix_chance_by_size["mask_size_class"]
    .map({name: index for index, name in enumerate(size_order)})
)

appendix_chance_by_size = (
    appendix_chance_by_size
    .sort_values(["Concept", "size_order", "Method"])
    [
        [
            "Concept",
            "Size Tier",
            "Method",
            "mean_observed_dice",
            "mean_dice_enrichment",
        ]
    ]
    .rename(
        columns={
            "mean_observed_dice": "Raw Dice",
            "mean_dice_enrichment": "Dice Enrichment",
        }
    )
)

appendix_chance_by_size[
    ["Raw Dice", "Dice Enrichment"]
] = appendix_chance_by_size[
    ["Raw Dice", "Dice Enrichment"]
].round(3)

display(appendix_chance_by_size)

### 3.4 SHAP colour heterogeneity enrichment by diagnostic label

In [ ]:
chance_by_label = pd.read_csv(
    RESULTS_DIR / "phase5_local_1260_260722_chance_summary_by_label.csv"
)

shap_colour_by_label = chance_by_label[
    (chance_by_label["xai_method"] == "shap")
    & (chance_by_label["concept"] == "colour_heterogeneity")
][
    [
        "label_name",
        "mean_dice_enrichment",
        "mean_sir_enrichment",
    ]
].copy()

shap_colour_by_label.columns = [
    "Diagnostic Group",
    "Dice Enrichment",
    "SIR Enrichment",
]

shap_colour_by_label["Diagnostic Group"] = (
    shap_colour_by_label["Diagnostic Group"]
    .replace({
        "melanoma": "Melanoma",
        "non-melanoma": "Non-Melanoma",
    })
)

shap_colour_by_label[
    ["Dice Enrichment", "SIR Enrichment"]
] = shap_colour_by_label[
    ["Dice Enrichment", "SIR Enrichment"]
].round(2)

display(shap_colour_by_label)